# Part 1: Ground answers in the web with Web IQ

Part 1 at Caldova begins with a temperature excursion. Shipment `SHIP-VC-1026-BOS-FRA` ran above its labelled storage range, and before the distribution team reviews Caldova's own records, it wants to know what current industry guidance says about handling an excursion. Caldova's documents can't answer that. You need fresh, citable content from the public web.

**🎯 Mission**
- Connect to the **Web IQ MCP server** with your API key and inspect the tools it exposes
- Search **web pages** for current cold-chain guidance and read the grounding passages
- Search **images** and **videos** to see how Web IQ covers other content types

**Before you run:** Follow [Exercise 1 in the lab guide](../../instructions/overview.md#exercise-1-web-iq) to create your Web IQ API key and save it in `.env` as `WEB_IQ_KEY`. Then follow this notebook from top to bottom, running the code cells in sequence.

## The building blocks

- **Web IQ**: AI-native APIs that return fresh, real-world content from web pages, news, images, videos, and places. Results are shaped for grounding a model, not for a search results page.
- **MCP endpoint**: Web IQ speaks Model Context Protocol (MCP) at `https://api.microsoft.ai/v3/mcp`. You authenticate by sending your API key in the `x-apikey` header.
- **Passages**: With `contentFormat` set to `passage`, each web result returns the part of the page that is most relevant to your query. Passages are the recommended format for grounding.

## Step 1: Load configuration and create an MCP session helper

Load the Web IQ key from the project `.env` file and define a helper that opens an authenticated MCP session.

**ℹ️ Note**
- The first time you run the cell below, you'll be prompted to select a kernel. Select **Python Environments** and choose the **.venv** environment.
- You may also be prompted with "Do you want to allow public and private networks to access this app?" Select **Allow**.

**⚠️ Troubleshooting:**
If code cells get stuck and keep spinning, try these remediation steps:
1. Select **Restart** from the notebook toolbar at the top. If that doesn't work:
2. Select **Reload window** from the VS Code command palette. If that doesn't work:
3. Close VS Code completely and restart it.

In [ ]:
import os
from contextlib import asynccontextmanager

import httpx
from dotenv import load_dotenv
from mcp import ClientSession
from mcp.client.streamable_http import streamable_http_client

load_dotenv(override=True)

WEB_IQ_KEY = os.getenv("WEB_IQ_KEY", "").strip()
if not WEB_IQ_KEY:
    raise RuntimeError(
        "WEB_IQ_KEY is missing from .env. Create an API key in the Web IQ portal "
        "as described in Exercise 1, add it to .env, save the file, and rerun this cell."
    )
WEB_IQ_MCP_URL = "https://api.microsoft.ai/v3/mcp"


@asynccontextmanager
async def open_webiq_session():
    async with httpx.AsyncClient(
        headers={"x-apikey": WEB_IQ_KEY},
        timeout=httpx.Timeout(30, read=120),
    ) as http_client:
        async with streamable_http_client(WEB_IQ_MCP_URL, http_client=http_client) as (read, write, _):
            async with ClientSession(read, write) as session:
                await session.initialize()
                yield session


print("Web IQ configuration loaded")

## Step 2: List the available tools

Open an MCP session and inspect every tool that Web IQ exposes. Each tool covers a different kind of web content, such as web pages, news, images, and videos. Your Web IQ profile controls which tools you can call. The lab profile allows web, image, and video search.

In [ ]:
from rich.console import Console

console = Console()

async with open_webiq_session() as session:
    tools_response = await session.list_tools()

for tool in tools_response.tools:
    console.print(f"[bold cyan]{tool.name}[/]: {tool.description}")
    console.print(f"  Parameters: {', '.join(tool.inputSchema.get('properties', {}))}")

## Step 3: Search the web for cold-chain guidance

Call the `web` tool with a query about handling temperature excursions. Each result has a title, a URL, and a passage. Look at where each result comes from. A peer-reviewed journal or a regulator is stronger evidence than a vendor blog.

Change `query` to try other searches, such as guidance for a specific market.

In [ ]:
query = "good distribution practice guidance for pharmaceutical temperature excursions during shipment"

async with open_webiq_session() as session:
    result = await session.call_tool("web", {"query": query, "maxResults": 5, "contentFormat": "passage"})

if result.isError:
    raise RuntimeError(f"Web IQ web search failed: {result.content}")

web_results = result.structuredContent["webResults"]
for number, page in enumerate(web_results, 1):
    console.print(f"[bold][{number}] {page.get('title', '')}[/]")
    console.print(f"[link={page['url']}]{page['url']}[/link]")
    console.print(page.get("content", "")[:400] + "...\n")

## Step 4: Search for images

The `images` tool returns images with a caption and the page that hosts each image. An agent can use the captions to pick an image that matches a request, such as a diagram of a temperature data logger for a training note.

In [ ]:
query = "pharmaceutical shipment temperature data logger"

async with open_webiq_session() as session:
    result = await session.call_tool("images", {"query": query, "maxResults": 5})

if result.isError:
    raise RuntimeError(f"Web IQ image search failed: {result.content}")

for image in result.structuredContent["imageResults"]:
    console.print(f"[bold]{image.get('title', '')}[/]")
    console.print(image.get("caption", ""))
    console.print(f"[link={image['hostPageUrl']}]{image['hostPageUrl']}[/link]\n")

## Step 5: Search for videos

The `videos` tool returns videos with their publisher, length, and view count. Use it to find training material on investigating a cold-chain excursion.

In [ ]:
query = "how to investigate a cold chain temperature excursion pharmaceutical"

async with open_webiq_session() as session:
    result = await session.call_tool("videos", {"query": query, "maxResults": 5})

if result.isError:
    raise RuntimeError(f"Web IQ video search failed: {result.content}")

for video in result.structuredContent["videoResults"]:
    console.print(f"[bold]{video.get('title', '')}[/]")
    console.print(f"{video.get('publishedBy', '')} | {video.get('length', '')} | {video.get('viewCount', 0):,} views")
    console.print(f"[link={video['url']}]{video['url']}[/link]\n")

#### 🔍 Source Hunt

Look at the web results from Step 3. Can you find:

1. Which results come from a peer-reviewed journal or a regulator, and which come from vendors?
2. Does each passage support a specific statement you could cite in a briefing? Open one URL in the browser and find the passage on the page.
3. Which questions about shipment `SHIP-VC-1026-BOS-FRA` can the web **not** answer? Keep them in mind. You will answer them from Caldova's own evidence in Part 4.

## ✅ Mission complete

**What you built:**

* ✓ **Authenticated MCP session**: A direct connection to the Web IQ MCP endpoint using your API key in the `x-apikey` header.
* ✓ **Grounded web search**: Current cold-chain guidance returned as passages with source URLs, ready to cite.
* ✓ **Image and video search**: Results from the other Web IQ tools, each with the page or publisher it came from.

Web IQ gives an agent public context. It cannot see Caldova's lakehouse, mailbox, or documents. The next parts add those sources.

➡️ Select **Next** in the lab sidebar to start the [Exercise 2 browser exploration](../../instructions/overview.md#exercise-2-fabric-iq), then open [Part 2: Query structured data with Fabric IQ](part2-fabric-iq.ipynb). Outside Skillable, use the exercise link.